In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import pandas as pd
df = pd.read_parquet('data/raw/part-00000.parquet')
df.shape

(200000, 2)

In [3]:
def info(ds, test=False):
    if test:
        return [
            ds.question.str.len().mean(),
            None,
            ds.question.str.startswith('Student:').mean(),
            None,
            None
        ]
    return [
        ds.question.str.len().mean(),
        ds.code.str.len().mean(),
        ds.question.str.startswith('Student:').mean(),
        ds.code.str.contains('print(', regex=False).mean(),
        ds.code.str.contains('#').mean()
    ]

In [4]:
matrix = []
parts = ['part-00000.parquet', 'part-00013.parquet', 'part-00020.parquet', 'part-00033.parquet', 'part-00045.parquet', 'part-00055.parquet', 'test.parquet']

for part in parts:
    ds = pd.read_parquet(f'data/raw/{part}')
    if part == 'test.parquet':
        matrix.append([part, *info(ds, test=True)])
    else:
        matrix.append([part, *info(ds)])
    
pd.DataFrame(matrix, columns=['part', 'avg_question_len', 'avg_code_len', 'question_starts_with_student', 'code_contains_print', 'code_contains_comment'])


,part,avg_question_len,avg_code_len,question_starts_with_student,code_contains_print,code_contains_comment
0,part-00000.parquet,177.697005,236.204995,0.128650,0.000050,0.125815
1,part-00013.parquet,177.534370,235.559125,0.127335,0.000045,0.125220
2,part-00020.parquet,167.249625,220.996250,0.000000,0.000515,0.151385
3,part-00033.parquet,219.074900,314.369030,0.000000,0.903680,0.865840
4,part-00045.parquet,166.818905,220.667165,0.000000,0.000500,0.149900
5,part-00055.parquet,297.278655,408.356775,0.000000,0.888075,0.834420
6,test.parquet,177.663366,NaN,0.139656,NaN,NaN


без fork

In [5]:
from src.execute import run, run_many
import time

codes = df.code[:300].tolist()
t = time.time()
a = [run(c) for c in codes]
print(time.time() - t)
t = time.time()
b = run_many(codes)
print(time.time() - t)

3.9765713214874268
0.8534550666809082


с fork

In [6]:
from src.execute import run, run_many
import time

codes = df.code[:300].tolist()
t = time.time()
a = [run(c) for c in codes]
print(time.time() - t)
t = time.time()
b = run_many(codes)
print(time.time() - t)
a == b

3.4811630249023438
1.0787229537963867


True

In [7]:
from src.data import build_val
from src.execute import pass1
import json

val_df = build_val('data/raw/part-00013.parquet', 'data/val.jsonl')

val = [json.loads(l) for l in open('data/val.jsonl')]
golds = [v['gold'] for v in val]
print(len(val))
print(pass1([v['code'] for v in val], golds))
print(pass1([''] * len(val), golds))


2000
1.0
0.0


In [8]:
from src.normalize import normalize, denormalize
from src.execute import run_many, is_correct

def check_shard(path, n=3000, seed=0):
    df = pd.read_parquet(path).sample(n, random_state=seed)
    codes = df.code.tolist()
    bodies = [normalize(c) for c in codes]

    orig = run_many(codes)
    back = run_many([denormalize(b) if b is not None else '' for b in bodies])

    valid = [i for i in range(n) if orig[i] is not None and bodies[i] is not None]
    agree = sum(is_correct(back[i], orig[i]) for i in valid)

    len_before = sum(len(c) for c in codes) / n
    len_after = sum(len(b) for b in bodies if b is not None) / len([b for b in bodies if b is not None])
    print(f'{path}: None {bodies.count(None)}, совпало {agree}/{len(valid)}, '
          f'длина {len_before:.0f} -> {len_after:.0f}')
    return codes, bodies, orig, back

for shard in ['00000', '00020', '00055']:
    check_shard(f'data/raw/part-{shard}.parquet')


data/raw/part-00000.parquet: None 0, совпало 2883/2883, длина 237 -> 156
data/raw/part-00020.parquet: None 1, совпало 2907/2907, длина 221 -> 147
data/raw/part-00055.parquet: None 0, совпало 2781/2781, длина 404 -> 250


In [9]:
from src.normalize import normalize
from src.tokenizer import train_tokenizer, measure

df0 = pd.read_parquet('data/raw/part-00000.parquet')
df0['body'] = [normalize(c) for c in df0.code]
df0 = df0[df0.body.notna()]
fit, ev = df0.iloc[:150000], df0.iloc[150000:160000]
texts = fit.question.tolist() + fit.body.tolist()

results = {}
for kind in ['gpt2', 'noregex', 'numbers', 'digits']:
    tok = train_tokenizer(texts, kind)
    results[kind] = measure(tok, ev.question.tolist(), ev.body.tolist())
    print(kind, tok.encode(ev.body.iloc[1]).tokens[:12])
pd.DataFrame(results).T


gpt2 ['yellow', 'B', 'alls', 'Ġ=', 'Ġ6', 'Ċ', 'blue', 'B', 'alls', 'Ġ=', 'Ġ4', 'Ċ']
noregex ['yellow', 'B', 'all', 'sĠ=Ġ6Ċ', 'blue', 'B', 'all', 'sĠ=Ġ4Ċ', 'total', 'B', 'all', 'sĠ=Ġ']
numbers ['yellow', 'B', 'all', 'sĠ=Ġ', '6', 'Ċ', 'blue', 'B', 'all', 'sĠ=Ġ', '4', 'Ċtotal']
digits ['yellow', 'B', 'all', 'sĠ=Ġ', '6', 'Ċ', 'blue', 'B', 'all', 'sĠ=Ġ', '4', 'Ċtotal']


,vocab,code_mean,code_p95,code_le64,q_mean,roundtrip_bad
gpt2,16000.0,50.9,104.0,0.756,41.8,0.0
noregex,16000.0,26.6,58.0,0.968,27.7,0.0
numbers,16000.0,29.9,64.0,0.951,29.5,0.0
digits,16000.0,32.6,69.0,0.937,31.8,0.0


In [10]:
results = {}
for vocab_size in [8000, 16000, 32000]:
    tok = train_tokenizer(texts, 'numbers', vocab_size=vocab_size)
    results[vocab_size] = measure(tok, ev.question.tolist(), ev.body.tolist())
pd.DataFrame(results).T

,vocab,code_mean,code_p95,code_le64,q_mean,roundtrip_bad
8000,8000.0,33.8,73.0,0.923,34.2,0.0
16000,16000.0,29.9,64.0,0.951,29.5,0.0
32000,32000.0,27.0,58.0,0.969,26.0,0.0


In [11]:
from tokenizers import Tokenizer
from src.normalize import normalize
from src.tokenizer import train_tokenizer, measure

texts = []
for shard in ['00000', '00001', '00002', '00020', '00055']:
    df = pd.read_parquet(f'data/raw/part-{shard}.parquet').sample(60000, random_state=0)
    bodies = [normalize(c) for c in df.code]
    texts += df.question.tolist()
    texts += [b for b in bodies if b is not None]

print(len(texts))
tok = train_tokenizer(texts, 'numbers', vocab_size=16000)
tok.save('data/tokenizer.json')

599998


In [12]:
tok = Tokenizer.from_file('data/tokenizer.json')
measure(tok, ev.question.tolist(), ev.body.tolist())

{'vocab': 16000,
 'code_mean': 30.4,
 'code_p95': 65,
 'code_le64': 0.949,
 'q_mean': 30.1,
 'roundtrip_bad': 0}

In [13]:
import os, json
from tokenizers import Tokenizer
from src.data import encode_shard

tok = Tokenizer.from_file('data/tokenizer.json')
exclude = {json.loads(l)['question'] for l in open('data/val.jsonl')}
exclude |= set(pd.read_parquet('data/raw/test.parquet').question)
os.makedirs('data/enc', exist_ok=True)

stats = []
for shard in ['00000', '00001', '00002', '00013', '00020', '00033', '00045', '00055', '00059']:
    stats.append(encode_shard(f'data/raw/part-{shard}.parquet', f'data/enc/part-{shard}.npz', tok, exclude))
pd.DataFrame(stats)

{'shard': 'data/raw/part-00000.parquet', 'rows': 200000, 'val_test': 46, 'normalize_none': 0, 'too_long': 476, 'kept': 199478}
{'shard': 'data/raw/part-00001.parquet', 'rows': 200000, 'val_test': 43, 'normalize_none': 0, 'too_long': 453, 'kept': 199504}
{'shard': 'data/raw/part-00002.parquet', 'rows': 200000, 'val_test': 55, 'normalize_none': 1, 'too_long': 454, 'kept': 199490}
{'shard': 'data/raw/part-00013.parquet', 'rows': 200000, 'val_test': 2036, 'normalize_none': 0, 'too_long': 461, 'kept': 197503}
{'shard': 'data/raw/part-00020.parquet', 'rows': 200000, 'val_test': 15, 'normalize_none': 2, 'too_long': 119, 'kept': 199864}
{'shard': 'data/raw/part-00033.parquet', 'rows': 200000, 'val_test': 0, 'normalize_none': 2, 'too_long': 302, 'kept': 199696}
{'shard': 'data/raw/part-00045.parquet', 'rows': 200000, 'val_test': 4, 'normalize_none': 2, 'too_long': 124, 'kept': 199870}
{'shard': 'data/raw/part-00055.parquet', 'rows': 200000, 'val_test': 0, 'normalize_none': 4, 'too_long': 1405, 

,shard,rows,val_test,normalize_none,too_long,kept
0,data/raw/part-00000.parquet,200000,46,0,476,199478
1,data/raw/part-00001.parquet,200000,43,0,453,199504
2,data/raw/part-00002.parquet,200000,55,1,454,199490
3,data/raw/part-00013.parquet,200000,2036,0,461,197503
4,data/raw/part-00020.parquet,200000,15,2,119,199864
5,data/raw/part-00033.parquet,200000,0,2,302,199696
6,data/raw/part-00045.parquet,200000,4,2,124,199870
7,data/raw/part-00055.parquet,200000,0,4,1405,198591
8,data/raw/part-00059.parquet,38003,0,3,247,37753


In [14]:
from src.batches import Data
d = Data(['data/enc/part-00000.npz', 'data/enc/part-00001.npz'])
print(d.n)
print(d.q_len.min(), d.c_len.min())
print(d.q_off[-1] == len(d.q), d.c_off[-1] == len(d.c))

398982
3 3
True True


In [15]:
from tokenizers import Tokenizer
tok = Tokenizer.from_file('data/tokenizer.json')
i = 199478
print(tok.decode(d.q[d.q_off[i]:d.q_off[i+1]].tolist()))
print(tok.decode(d.c[d.c_off[i]:d.c_off[i+1]].tolist()))

Jane has 12 colored pencils. She has 3 times as many markers as pencils. How many markers does Jane have in total?
pencils = 12
markers = pencils*3
result = markers


In [16]:
import numpy as np
rng = np.random.default_rng(0)

idx = next(d.batches(256, rng))
src, tgt_in, tgt_out = d.collate(idx)
print(src.shape, tgt_in.shape, tgt_out.shape)
mask = tgt_out[:, 1:] != -100
print('сдвиг верный:', bool((tgt_in[:, 1:][mask] == tgt_out[:, :-1][mask]).all()))
print('первый токен везде BOS:', bool((tgt_in[:, 0] == 1).all()))

def pad_share(batches):
    pad = total = 0
    for idx in batches:
        s, ti, _ = d.collate(idx)
        pad += (s == 0).sum().item() + (ti == 0).sum().item()
        total += s.numel() + ti.numel()
    return pad / total

stream = d.batches(256, rng)
bucketed = [next(stream) for _ in range(100)]
random_ = [rng.choice(d.n, 256, replace=False) for _ in range(100)]
print(f'паддинг: bucketing {pad_share(bucketed):.1%}, случайные {pad_share(random_):.1%}')


torch.Size([256, 24]) torch.Size([256, 40]) torch.Size([256, 40])
сдвиг верный: True
первый токен везде BOS: True
паддинг: bucketing 13.8%, случайные 72.8%


In [17]:
from src.model import Config, Attention
import torch

cfg = Config(d=64, heads=4)
att = Attention(cfg)
x, mem = torch.randn(2, 5, 64), torch.randn(2, 7, 64)
print(att(x).shape, att(x, mem=mem).shape)

x2 = x.clone(); x2[:, 3:] = torch.randn(2, 2, 64)
print(torch.allclose(att(x, causal=True)[:, :3], att(x2, causal=True)[:, :3], atol=1e-6))

mask = torch.ones(2, 1, 1, 7, dtype=torch.bool); mask[1, :, :, 5:] = False
mem2 = mem.clone(); mem2[1, 5:] = torch.randn(2, 64)
print(torch.allclose(att(x, mem=mem, mask=mask), att(x, mem=mem2, mask=mask), atol=1e-6))

torch.Size([2, 5, 64]) torch.Size([2, 5, 64])
True
True


In [18]:
from src.model import EncoderLayer, DecoderLayer

cfg = Config(d=64, heads=4)
x, mem = torch.randn(2, 5, 64), torch.randn(2, 7, 64)
mask = torch.ones(2, 1, 1, 7, dtype=torch.bool)

print(EncoderLayer(cfg)(mem, mask).shape)
print(DecoderLayer(cfg)(x, mem, mask).shape)

n = lambda m: sum(p.numel() for p in m.parameters())
print(n(EncoderLayer(Config())), 12 * 512**2)
print(n(DecoderLayer(Config())), 16 * 512**2)

dec = DecoderLayer(cfg)
x2 = x.clone(); x2[:, 3:] = torch.randn(2, 2, 64)
print(torch.allclose(dec(x, mem, mask)[:, :3], dec(x2, mem, mask)[:, :3], atol=1e-6))   # True


torch.Size([2, 7, 64])
torch.Size([2, 5, 64])
3147776 3145728
4197376 4194304
True


In [19]:
from src.model import Config, MishaGustokashin
from src.batches import Data
import numpy as np, torch

d = Data(['data/enc/part-00000.npz'])
src, tgt_in, tgt_out = d.collate(next(d.batches(256, np.random.default_rng(0))))

model = MishaGustokashin(Config(d=64, heads=4, n_enc=2, n_dec=2))
logits = model(src, tgt_in)
print(logits.shape, tgt_out.shape)

loss = torch.nn.functional.cross_entropy(logits.view(-1, 16000), tgt_out.view(-1))
print(loss.item())

torch.Size([256, 16, 16000]) torch.Size([256, 16])
9.699668884277344


In [20]:
import torch.nn.functional as F

torch.manual_seed(0)
idx = np.arange(64)
src, tgt_in, tgt_out = d.collate(idx)

model = MishaGustokashin(Config(d=128, heads=4, n_enc=2, n_dec=2))
opt = torch.optim.AdamW(model.parameters(), lr=1e-3)

for step in range(300):
    logits = model(src, tgt_in)
    loss = F.cross_entropy(logits.view(-1, logits.shape[-1]), tgt_out.view(-1))
    opt.zero_grad()
    loss.backward()
    opt.step()
    if step % 50 == 0 or step == 299:
        print(step, round(loss.item(), 4))


0 9.717
50 4.5494
100 1.3383
150 0.1321
200 0.0442
250 0.0251
299 0.017


In [21]:
import json, torch
from tokenizers import Tokenizer
from src.generate import load_model
from src.vote import candidates, run_groups, vote
from src.execute import is_correct

device = torch.device('cpu')
tok = Tokenizer.from_file('data/tokenizer.json')
model = load_model('runs/base_d512_6x6/best.pt', device)
val = [json.loads(l) for l in open('data/val.jsonl')]


def show(i, k=8, T=0.7):
    q, gold = val[i]['question'], val[i]['gold']
    cs = candidates(model, tok, [q], device, k, temperature=T)[0]
    vs = run_groups([[c for c, _ in cs]])[0]
    code, value = vote(cs, vs)

    print(q, '\nверный ответ:', gold, '\n')
    for j, ((c, lp), v) in enumerate(zip(cs, vs)):
        mark = '✓' if is_correct(v, gold) else '✗'
        kind = 'жадный' if j == 0 else 'сэмпл '
        print(f'#{j} {kind}  ответ={v}  {mark}  logp={lp:.2f}')
        print('    ' + '\n    '.join(c.split('\n')[1:-2]))
    print('\nголосование выбрало:', value, '✓' if is_correct(value, gold) else '✗')


show(100)

A pack of 24 markers costs $25. Jane buys 3 packs of markers. How much money does Jane pay in total? 
верный ответ: 75.0 

#0 жадный  ответ=75.0  ✓  logp=-4.78
        markers_per_pack = 24
        pack_price = 25
        packs_bought = 3
        total_price = pack_price * packs_bought
        result = total_price
#1 сэмпл   ответ=75.0  ✓  logp=-4.80
        markers_per_pack = 24
        pack_price = 25
        num_packs = 3
        total_price = pack_price * num_packs
        result = total_price
#2 сэмпл   ответ=75.0  ✓  logp=-5.21
        markers_per_pack = 24
        pack_cost = 25
        packs_bought = 3
        total_cost = pack_cost * packs_bought
        result = total_cost
#3 сэмпл   ответ=1800.0  ✗  logp=-6.69
        num_packs = 3
        markers_per_pack = 24
        pack_cost = 25
        total_cost = num_packs * markers_per_pack * pack_cost
        result = total_cost
#4 сэмпл   ответ=75.0  ✓  logp=-5.06
        pack_size = 24
        pack_price = 25
        num_packs = 

In [ ]:
import json, torch
from tokenizers import Tokenizer
from src.generate import load_model
from src.model import BOS, EOS

tok = Tokenizer.from_file('data/tokenizer.json')
model = load_model('runs/base_d512_6x6/best.pt', torch.device('cpu'))
q = json.loads(open('data/val.jsonl').readline())['question']
print(q, '\n')

src = torch.tensor([tok.encode(q).ids])
ys = torch.tensor([[BOS]])
with torch.no_grad():
    for _ in range(60):
        probs = model(src, ys)[0, -1].softmax(-1)
        top = probs.topk(5)
        nxt = top.indices[0].item()
        alts = '  '.join(f'{tok.decode([i])!r}:{p:.2f}' for p, i in zip(top.values.tolist(), top.indices.tolist()))
        print(f'{tok.decode([nxt])!r:18} | {alts}')
        if nxt == EOS:
            break
        ys = torch.cat([ys, torch.tensor([[nxt]])], 1)

Kyle is driving from his home to a lake and will drive 102 miles. He averages 25 miles per hour. How many hours will it take him to get there? 

'distance = '      | 'distance = ':0.51  'total_distance = ':0.10  'total_miles = ':0.07  'miles = ':0.07  'miles_':0.02
'10'               | '10':1.00  '0':0.00  '1':0.00  '5':0.00  '10000':0.00
'2'                | '2':1.00  '1':0.00  '4':0.00  '0':0.00  '52':0.00
'\nspeed = '       | '\nspeed = ':0.67  '\nm':0.10  '\naverage_speed = ':0.07  '\navg_':0.04  '\n':0.03
'25'               | '25':1.00  '0':0.00  ' ':0.00  '75':0.00  '50':0.00
'\nresult = '      | '\nresult = ':0.29  '\ntime_':0.14  '\ntime = distance/':0.13  '\nhours = ':0.09  '\ntime_taken = ':0.06
'distance / '      | 'distance / ':0.69  'distance/':0.25  'distance ':0.05  'distance':0.00  '0':0.00
'speed'            | 'speed':0.99  'speed\nresult = ':0.01  '(':0.00  'speed ':0.00  'speed * ':0.00
''                 | '':0.99  '\np':0.01  '\nresult = ':0.00  '\nresult = round('